In [4]:
import sys
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from awsglue.dynamicframe import DynamicFrame
from pyspark.sql.functions import current_timestamp, lit

# --- 1. CONFIGURATION & SETTINGS ---
catalog_name  = "glue_catalog"
database_name = "processdb_iceberg"
source_db     = "landingdb"
source_table  = "formulaone_circuits_csv"
warehouse_path = "s3://formulaone-glue/process_iceberg/"

# Clean file name for the audit column (formulaone-circuits_csv -> circuits.csv)
file_name_audit = source_table.replace("formulaone-", "").replace("_", ".")
target_table_path = f"{catalog_name}.{database_name}.circuits"

# --- 2. SPARK & ICEBERG INITIALIZATION ---
conf = SparkConf()

# Initialize Contexts
sc = SparkContext.getOrCreate(conf=conf)
glueContext = GlueContext(sc)
spark = glueContext.spark_session

print(f"Initialised. Catalog '{catalog_name}' is ready to talk to AWS Glue.")

# --- 3. READ DATA FROM LANDING ---
print(f"Reading from {source_db}.{source_table}...")
circuits_dynamic_frame = glueContext.create_dynamic_frame.from_catalog(
    database=source_db,
    table_name=source_table
)

# WRITE TO S3 BUCKET
# glueContext.write_dynamic_frame.from_options(
#     frame = circuits_dynamic_frame,
#     connection_type = "s3",
#     connection_options = "",
#     format = "parquet"
# )

# job.commit()

# Convert to Spark DataFrame for transformation
df = circuits_dynamic_frame.toDF()

# --- 4. TRANSFORMATIONS ---
rename_map = {
    "circuitid": "circuit_id",
    "circuitRef": "circuit_ref",
    "lat": "latitude",
    "lng": "longitude",
    "alt": "altitude",
}

# Apply renames using a loop
for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

# Add Audit Columns and Drop 'url'
circuits_df = df.withColumn("load_timestamp", current_timestamp()) \
       .withColumn("file_name", lit(file_name_audit)) \
       .drop("url")

circuits_df.show(10, False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Initialised. Catalog 'glue_catalog' is ready to talk to AWS Glue.
Reading from landingdb.formulaone_circuits_csv...
+--------------+---------+
|load_timestamp|file_name|
+--------------+---------+
+--------------+---------+